In [0]:
from pyspark.sql import functions as F


# =========================================================
# CONFIGURATION
# =========================================================

SOURCE_PATH = (
    "/Volumes/retail_demo/raw/landing/sales/"
)

SCHEMA_LOCATION = (
    "/Volumes/retail_demo/raw/landing/"
    "_autoloader/sales/schema/"
)

CHECKPOINT_LOCATION = (
    "/Volumes/retail_demo/raw/landing/"
    "_autoloader/sales/checkpoint/"
)

TARGET_TABLE = (
    "retail_demo.bronze.sales_transactions"
)


# =========================================================
# CHECK SOURCE
# =========================================================

print("Source files:")

for file in dbutils.fs.ls(SOURCE_PATH):
    print(file.name)


# =========================================================
# AUTO LOADER
# =========================================================

sales_stream = (
    spark.readStream
    .format("cloudFiles")

    .option(
        "cloudFiles.format",
        "csv"
    )

    .option(
        "cloudFiles.schemaLocation",
        SCHEMA_LOCATION
    )

    .option(
        "cloudFiles.inferColumnTypes",
        "true"
    )

    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )

    .option(
        "header",
        "true"
    )

    .option(
        "rescuedDataColumn",
        "_rescued_data"
    )

    .load(SOURCE_PATH)
)


# =========================================================
# BRONZE METADATA
# =========================================================

sales_bronze = (
    sales_stream

    .withColumn(
        "_source_file",
        F.col("_metadata.file_name")
    )

    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )

    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size")
    )

    .withColumn(
        "_source_file_modified",
        F.col("_metadata.file_modification_time")
    )

    .withColumn(
        "_bronze_ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_bronze_ingestion_date",
        F.current_date()
    )
)


# =========================================================
# PRINT SCHEMA
# =========================================================

sales_bronze.printSchema()


# =========================================================
# WRITE BRONZE
# =========================================================

query = (
    sales_bronze.writeStream

    .format("delta")

    .option(
        "checkpointLocation",
        CHECKPOINT_LOCATION
    )

    .option(
        "mergeSchema",
        "true"
    )

    .outputMode("append")

    .trigger(
        availableNow=True
    )

    .toTable(
        TARGET_TABLE
    )
)


# =========================================================
# WAIT FOR COMPLETION
# =========================================================

query.awaitTermination()

print("Sales Auto Loader completed successfully.")